In [1]:
from datetime import datetime, timedelta
import pandas as pd
import numpy as np

In [2]:
ENTREPOT_PATH = '/home/administrateur/Bureau/Datagrosyst/data_entrepot_outils/'
EXTERNAL_DATA_PATH = '/home/administrateur/Bureau/Datagrosyst/catalogue_script_agrosyst/02_outils/data/external_data/'
donnees = {}

def import_dfs(df_names, path_data, sep = ','):
    i = 0
    for df_name in df_names: 
        donnees[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, low_memory=False).replace({'\r\n': '\n'}, regex=True)

tables = [
    'composant_culture', 
    'culture',
    # 'espece',
    'intervention_synthetise',
    'intervention_realise',
    'connection_synthetise',
    'noeuds_synthetise_restructure',
    'noeuds_realise',

    'noeuds_synthetise',
    'synthetise',
    'zone',
    'parcelle',
    'sdc'
]

tables_ext = [
    'espece', # en réalité un fichier espece de MAE
    'matrice_passage_dirodur_famille_bota', 
    'matrice_passage_dirodur_espece_precise'
]

# import des données
import_dfs(tables, ENTREPOT_PATH, sep = ',')
import_dfs(tables_ext, EXTERNAL_DATA_PATH, sep = ',')

In [3]:
# donnees de base
cropsp = donnees['composant_culture'][['espece_id','culture_id','compagne']].copy()
crop = donnees['culture'][['id','nom','type']].rename(columns={'id':'culture_id'}).copy()
sp = donnees['espece'][['id','typodirodur_espece_precise','typodirodur_espece_famille_bota','typodirodur_espece_periode_semis']].rename(columns={'id':'espece_id'}).copy()

# Donnees d'intervention pour les dates de semis
intv_S = donnees['intervention_synthetise'][['id','type','date_debut','date_fin','concerne_ci','connection_synthetise_id']].copy()
intv_R = donnees['intervention_realise'][['id','type','date_debut','date_fin','concerne_ci','noeuds_realise_id']].copy()
conx_S = donnees['connection_synthetise'][['id','cible_noeuds_synthetise_id']].rename(columns={'id':'connection_synthetise_id', 'cible_noeuds_synthetise_id':'noeuds_synthetise_id'}).copy()
noeud_w_culture_id_S = donnees['noeuds_synthetise_restructure'][['id','culture_id']].rename(columns={'id':'noeuds_synthetise_id'}).copy()
noeuds_w_culture_id_R = donnees['noeuds_realise'][['id','culture_id','zone_id']].rename(columns={'id':'noeuds_realise_id'}).copy()

node_S = donnees['noeuds_synthetise'][['id','synthetise_id']].rename(columns={'id':'noeuds_synthetise_id'}).copy()
synthe = donnees['synthetise'][['id','sdc_id']].rename(columns={'id':'synthetise_id'}).copy()
zone = donnees['zone'][['id','parcelle_id']].rename(columns={'id':'zone_id'}).copy()
parcelle = donnees['parcelle'][['id','sdc_id']].rename(columns={'id':'parcelle_id'}).copy()
sdc = donnees['sdc'][['id','filiere']].rename(columns={'id':'sdc_id'}).copy()

In [4]:
intv_S = intv_S.loc[(intv_S['type'] == 'SEMIS') & 
                    (intv_S['connection_synthetise_id'].notna()) & 
                    (intv_S['concerne_ci'] == 'f')]
intv_R = intv_R.loc[(intv_R['type'] == 'SEMIS') & 
                    (intv_R['noeuds_realise_id'].notna()) & 
                    (intv_R['concerne_ci'] == 'f')]
intv_S = intv_S.merge(conx_S, on='connection_synthetise_id', how='left')
intv_S = intv_S.merge(noeud_w_culture_id_S, on='noeuds_synthetise_id', how='left')
intv_R = intv_R.merge(noeuds_w_culture_id_R, on='noeuds_realise_id', how='left')

In [5]:
def moyenne_dates(start_series, end_series, methode = 'R'):
    def parse_S_date(s):
        j, m = map(int, s.split('/'))
        try :
            return datetime(2025, m, j)
        except ValueError:
            try : 
                return datetime(2025, m, j-1)
            except ValueError:
                return datetime(2025, m, j-2)
    def parse_R_date(s):
        y, m, j = map(int, s.split('-'))
        return datetime(y, m, j)
    
    if methode == 'S':
        start_dates = start_series.apply(parse_S_date)
        end_dates = end_series.apply(parse_S_date)

        mask = end_dates < start_dates
        end_dates[mask] = end_dates[mask].apply(lambda x: datetime(2026, x.month, x.day))

    elif methode == 'R':
        start_dates = start_series.apply(parse_R_date)
        end_dates = end_series.apply(parse_R_date)

    moyennes = start_dates + ((end_dates - start_dates) / 2)

    return moyennes.dt.strftime('%d/%m')


intv_R['date_semis'] = moyenne_dates(intv_R['date_debut'], intv_R['date_fin'], methode='R')
intv_S['date_semis'] = moyenne_dates(intv_S['date_debut'], intv_S['date_fin'], methode='S')

In [6]:
intv = pd.concat([intv_S, intv_R], ignore_index=True)
intv = intv.merge(node_S, on='noeuds_synthetise_id',how='left')
intv = intv.merge(synthe, on='synthetise_id',how='left')
intv = intv.merge(zone, on='zone_id',how='left')
intv = intv.merge(parcelle, on='parcelle_id',how='left')

In [7]:
intv['sdc_id'] = np.where(intv['sdc_id_x'].notna(), intv['sdc_id_x'], intv['sdc_id_y'])
intv = intv[['id','culture_id','date_semis','sdc_id']]

intv = intv.merge(sdc, on='sdc_id',how='left')
intv = intv.loc[intv['filiere'].isin(['POLYCULTURE_ELEVAGE','GRANDES_CULTURES'])]

In [8]:
# Ajouter une année bissextile (2020) pour parser les dates
intv['_dayofyear'] = pd.to_datetime(
    intv['date_semis'].astype(str) + '/2020',
    format='%d/%m/%Y'
).dt.dayofyear

# Fonction pour trier un groupe en boucle circulaire
def tri_et_moy_date_semis(group):
    days = group['_dayofyear'].values
    if len(list(set(days))) == 1:
        return pd.Series({
            'dates_nbjr_triees': days.tolist(),
            'date_moyenne': group['date_semis'].values[0]
        })

    # Trouve l'écart maximal entre 2 dates consécutives (en boucle)
    sorted_days = np.sort(days)
    diffs = np.diff(np.r_[sorted_days, sorted_days[0] + 366])  # +366 pour gérer 29/02
    cut = np.argmax(diffs)+1
    # Réordonne en partant après la coupure
    order = np.roll(sorted_days, -cut)

    # Retourne la date moyenne
    asc_list = [y + order[0] if y < order[0] else y for y in order]
    moy = np.mean(asc_list)
    if moy > 366:
        moy -= 366
    date_moy = datetime(2020, 1, 1) + timedelta(days=moy-1)
    date_moy = date_moy.strftime('%d/%m') 

    return pd.Series({
        'dates_nbjr_triees': order.tolist(),
        'date_moyenne': date_moy
    })

# Applique le tri par groupe
semis = intv.groupby('culture_id', group_keys = False).apply(tri_et_moy_date_semis, include_groups=False)

In [9]:
# Donnees de typologie d'espece et de culture
typo1 = donnees['matrice_passage_dirodur_espece_precise'].copy()
typo2 = donnees['matrice_passage_dirodur_famille_bota'].copy()

df = cropsp.merge(sp, how = 'left', on = 'espece_id')

In [10]:
# Liste des cultures qui contiennent des cultures compagnes
list_culture_with_compagne = list(set(df.loc[df['compagne'].notnull(), 'culture_id']))

df['nb_composant_culture'] = 1
df['nb_typodirodur_espece_precise'] = df['typodirodur_espece_precise'].copy()
df['nb_typodirodur_espece_famille_bota'] = df['typodirodur_espece_famille_bota'].copy()
df['nb_typodirodur_espece_periode_semis'] = df['typodirodur_espece_periode_semis'].copy()

def concat_unique_sorted(series):
    cleaned = series.dropna().unique()
    if len(cleaned) == 0:
        return np.nan
    return '_'.join(sorted(cleaned))

def get_nb_unique_typo(series):
    cleaned = series.dropna().unique()
    return len(cleaned)

agg_dict = {
    'typodirodur_espece_precise': concat_unique_sorted,
    'typodirodur_espece_famille_bota': concat_unique_sorted,
    'typodirodur_espece_periode_semis': concat_unique_sorted,
    'nb_composant_culture': 'sum',
    'nb_typodirodur_espece_precise': get_nb_unique_typo,
    'nb_typodirodur_espece_famille_bota': get_nb_unique_typo,
    'nb_typodirodur_espece_periode_semis': get_nb_unique_typo,
    }

#  On crée les typologie can culture et les autre variable utiles grace a agg_dict
df = df[['culture_id',
              'typodirodur_espece_precise',
              'typodirodur_espece_famille_bota',
              'typodirodur_espece_periode_semis',
              'nb_composant_culture',
              'nb_typodirodur_espece_precise',
              'nb_typodirodur_espece_famille_bota',
              'nb_typodirodur_espece_periode_semis'
              ]].groupby('culture_id').agg(agg_dict).reset_index()


In [11]:
# Détection des cultures qui contiennent des cultures compagnes
df['is_any_compagne'] = np.where(df['culture_id'].isin(list_culture_with_compagne), True, False)

In [12]:
crop_only = crop.loc[~crop['culture_id'].isin(df['culture_id']),:]
crop_only.loc[:,['nb_composant_culture','nb_typodirodur_espece_precise','nb_typodirodur_espece_famille_bota','nb_typodirodur_espece_periode_semis']] = 0
crop_only.loc[:,['typodirodur_espece_precise','typodirodur_espece_famille_bota','typodirodur_espece_periode_semis','typocan_esp_precise_sans_compagne']] = 'espece_non_saisie'
crop_only['is_any_compagne'] = False

df = pd.concat([df, crop_only], ignore_index=True)

/tmp/ipykernel_289446/487873980.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  crop_only['is_any_compagne'] = False


In [13]:
def get_season(date_str):
    j, m = map(int, date_str.split('/'))
    date = datetime(2024, m, j)
    if datetime(2024, 1, 16) <= date <= datetime(2024, 4, 15):
        return 'printemps'
    elif datetime(2024, 4, 16) <= date <= datetime(2024, 6, 15):
        return 'ete'
    elif datetime(2024, 6, 16) <= date <= datetime(2024, 9, 15):
        return 'automne'
    else:
        return 'hiver'

# ATTENTION, on fait un merge right, ce qui permet de filtrer les cultures qui ne sont pas GCPE
df = df.merge(semis, how = 'right', on = 'culture_id')
df['saison_semis'] = df['date_moyenne'].apply(get_season)

In [14]:
df = df.merge(typo1, how='left', on='typodirodur_espece_precise')

# df = df.merge(typo2, how='left', on='typodirodur_espece_famille_bota')

# df = df.merge(typo1[['typodirodur_espece_precise','typodirodur_culture']].rename(columns={
#     'typodirodur_espece_precise': 'typocan_esp_precise_sans_compagne'
#     }), 
#     how='left', 
#     on='typocan_esp_precise_sans_compagne', 
#     suffixes=[None,'_sans_compagne'])

df['typodirodur_culture'] = np.where(df['nb_typodirodur_espece_precise'] == 1, df['typodirodur_espece_precise'], df['typodirodur_culture'])
df['est_culture_avec_compagne'] = np.where(df['nb_typodirodur_espece_precise'] == 1, 'non', df['est_culture_avec_compagne'])
# df['typodirodur_culture_sans_compagne'] = np.where(df['nb_typodirodur_espece_precise'] == 1, df['typodirodur_espece_precise'], df['typodirodur_culture_sans_compagne'])

df.loc[df['nb_composant_culture'] == 0,
       ['typodirodur_culture']] = 'espece_non_saisie'
df.loc[(df['nb_composant_culture'] > 0) & (df['nb_typodirodur_espece_precise'] == 0),
       ['typodirodur_culture']] = 'aucune_typo_espece_reconnue'
df.loc[(df['nb_composant_culture'] > 0) & (df['nb_typodirodur_espece_precise'] > 0) & (df['typodirodur_culture'].isna()),
       ['typodirodur_culture']] = 'aucune_correspondance_avec_culture'

In [ ]:
df.loc[df['typodirodur_culture']=='aucune_typo_espece_reconnue']

,culture_id,typodirodur_espece_precise,typodirodur_espece_famille_bota,typodirodur_espece_periode_semis,nb_composant_culture,nb_typodirodur_espece_precise,nb_typodirodur_espece_famille_bota,nb_typodirodur_espece_periode_semis,is_any_compagne,nom,type,typocan_esp_precise_sans_compagne,dates_nbjr_triees,date_moyenne,saison_semis,typodirodur_culture,est_culture_avec_compagne,est_culture_annuelle_asso,est_prairie,culture_periode_semis
18932,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,NaN,NaN,NaN,1,0,0,0,False,NaN,NaN,NaN,[107],16/04,ete,aucune_typo_espece_reconnue,NaN,NaN,NaN,NaN


In [19]:
df.loc[df['typodirodur_culture']=='espece_non_saisie']

,culture_id,typodirodur_espece_precise,typodirodur_espece_famille_bota,typodirodur_espece_periode_semis,nb_composant_culture,nb_typodirodur_espece_precise,nb_typodirodur_espece_famille_bota,nb_typodirodur_espece_periode_semis,is_any_compagne,nom,type,typocan_esp_precise_sans_compagne,dates_nbjr_triees,date_moyenne,saison_semis,typodirodur_culture,est_culture_avec_compagne,est_culture_annuelle_asso,est_prairie,culture_periode_semis
533,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,espece_non_saisie,espece_non_saisie,espece_non_saisie,0,0,0,0,False,TREFLE D'ALEXANDRIE,MAIN,espece_non_saisie,[249],05/09,automne,espece_non_saisie,NaN,NaN,NaN,NaN
872,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,espece_non_saisie,espece_non_saisie,espece_non_saisie,0,0,0,0,False,Luzerne,CATCH,espece_non_saisie,[188],06/07,automne,espece_non_saisie,NaN,NaN,NaN,NaN
1800,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,espece_non_saisie,espece_non_saisie,espece_non_saisie,0,0,0,0,False,TRITICALE D'HIVER,MAIN,espece_non_saisie,[284],10/10,hiver,espece_non_saisie,NaN,NaN,NaN,NaN
1944,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,espece_non_saisie,espece_non_saisie,espece_non_saisie,0,0,0,0,False,BLE TENDRE,MAIN,espece_non_saisie,[294],20/10,hiver,espece_non_saisie,NaN,NaN,NaN,NaN
2542,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,espece_non_saisie,espece_non_saisie,espece_non_saisie,0,0,0,0,False,Blé tendre d'hiver,MAIN,espece_non_saisie,[292],18/10,hiver,espece_non_saisie,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75212,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,espece_non_saisie,espece_non_saisie,espece_non_saisie,0,0,0,0,False,MAIS FOURRAGE,MAIN,espece_non_saisie,"[138, 138]",17/05,ete,espece_non_saisie,NaN,NaN,NaN,NaN
75615,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,espece_non_saisie,espece_non_saisie,espece_non_saisie,0,0,0,0,False,ORGE,MAIN,espece_non_saisie,[305],31/10,hiver,espece_non_saisie,NaN,NaN,NaN,NaN
76461,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,espece_non_saisie,espece_non_saisie,espece_non_saisie,0,0,0,0,False,TRITICALE D'HIVER,MAIN,espece_non_saisie,[285],11/10,hiver,espece_non_saisie,NaN,NaN,NaN,NaN
76466,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,espece_non_saisie,espece_non_saisie,espece_non_saisie,0,0,0,0,False,Maïs ensilage,MAIN,espece_non_saisie,[141],20/05,ete,espece_non_saisie,NaN,NaN,NaN,NaN


In [20]:
df.loc[df['typodirodur_culture']=='aucune_correspondance_avec_culture']

,culture_id,typodirodur_espece_precise,typodirodur_espece_famille_bota,typodirodur_espece_periode_semis,nb_composant_culture,nb_typodirodur_espece_precise,nb_typodirodur_espece_famille_bota,nb_typodirodur_espece_periode_semis,is_any_compagne,nom,type,typocan_esp_precise_sans_compagne,dates_nbjr_triees,date_moyenne,saison_semis,typodirodur_culture,est_culture_avec_compagne,est_culture_annuelle_asso,est_prairie,culture_periode_semis
417,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,Ray-grass d'Italie_Trèfle incarnat,Fabaceae_Poaceae,NaN,2,2,2,0,False,NaN,NaN,NaN,"[251, 251]",07/09,automne,aucune_correspondance_avec_culture,NaN,NaN,NaN,NaN
442,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,Ray-grass annuel_Trèfle incarnat,Fabaceae_Poaceae,NaN,2,2,2,0,False,NaN,NaN,NaN,[289],15/10,hiver,aucune_correspondance_avec_culture,NaN,NaN,NaN,NaN
555,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,Phacélie_Radis Daïkon = blanc Potager_Trèfle i...,Brassicaceae_Fabaceae_Hydrophyllaceae,NaN,3,3,3,0,False,NaN,NaN,NaN,[211],29/07,automne,aucune_correspondance_avec_culture,NaN,NaN,NaN,NaN
595,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,Ray-grass d'Italie_Trèfle d'Alexandrie,Fabaceae_Poaceae,NaN,2,2,2,0,False,NaN,NaN,NaN,"[239, 239, 239]",26/08,automne,aucune_correspondance_avec_culture,NaN,NaN,NaN,NaN
821,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,Ray-grass hybride_Trèfle d'Alexandrie,Fabaceae_Poaceae,NaN,2,2,2,0,False,NaN,NaN,NaN,"[294, 294]",20/10,hiver,aucune_correspondance_avec_culture,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76682,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,Ray-grass hybride_Trèfle d'Alexandrie,Fabaceae_Poaceae,NaN,2,2,2,0,False,NaN,NaN,NaN,[291],17/10,hiver,aucune_correspondance_avec_culture,NaN,NaN,NaN,NaN
77040,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,Ray-grass d'Italie_Trèfle incarnat,Fabaceae_Poaceae,NaN,2,2,2,0,False,NaN,NaN,NaN,"[275, 275, 275, 275, 275, 275]",01/10,hiver,aucune_correspondance_avec_culture,NaN,NaN,NaN,NaN
77238,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,Ray-grass d'Italie_Trèfle incarnat_Trèfle squa...,Fabaceae_Poaceae,NaN,4,4,2,0,False,NaN,NaN,NaN,"[231, 231]",18/08,automne,aucune_correspondance_avec_culture,NaN,NaN,NaN,NaN
77260,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,Ray-grass d'Italie_Trèfle incarnat,Fabaceae_Poaceae,NaN,2,2,2,0,False,NaN,NaN,NaN,[233],20/08,automne,aucune_correspondance_avec_culture,NaN,NaN,NaN,NaN


In [ ]:
df['typodirodur_culture'].value_counts(ascending=False)#.to_csv('/home/administrateur/Bureau/' + "test.csv", index=True)

In [ ]:
df['type'] = df['type'].astype('category')
df['type'] = df['type'].cat.rename_categories({'MAIN': 'PRINCIPALE', 
                                                'INTERMEDIATE': 'INTERMEDIAIRE', 
                                                'CATCH': 'DEROBEE' })
df['type'] = df['type'].astype('str')
df[['nb_composant_culture','nb_typocan_esp','nb_typocan_esp_maraich']] = df[['nb_composant_culture','nb_typocan_esp','nb_typocan_esp_maraich']].astype('int64')

# Ajout des tags 'culture porte-graines' dans une colonne à part, voir la détection quelques ligne plus tôt dans cette fonction
df = df.merge(culture_porteG[['culture_id','typo_cpg']], how='left', on='culture_id')

df